# 30B The learning loop
Priya's corrections become standing rules with provenance, and their effect is measured before and after. The stand-in follows a rule only when the rule is in its context, which makes the measurement visible; your agent's behavior is what you measure in the case study.

In [ ]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

In [ ]:
WS = workspace.fresh_workspace(ROOT / P["agent"]["seed"], ROOT / ".cache" / "nb30")   # a scratch copy: reruns start clean
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
led = audit.Ledger(clock=clock.clock)
tools = standin.read_tools(s, led, PANEL, MANDATES, M)
PLAN = [{"tool": "mandate_profile", "args": {"scope": "mandate_a"}, "rationale": "the client's positions"},
        {"tool": "bdc_signals", "args": {"scope": "public"}, "rationale": "what changed this release"},
        {"tool": "bdc_holdings", "args": {"scope": "public"}, "rationale": "terms behind each signal"}]
def note(rules_text):
    return loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "Standing rules:\n" + rules_text, clock.clock)[0]
CHECK = {   # one deterministic adherence check per correction
    "msg-101": lambda t: "PIK (watch)" not in t,
    "msg-102": lambda t: all("%" not in l for l in t.splitlines() if "marked" in l),
    "msg-103": lambda t: t.startswith("Your exposure"),
    "msg-104": lambda t: all("[0009" in l for l in t.splitlines() if ": " in l and not l.startswith(("Your exposure", "PIK"))),
}

In [ ]:
rows = []
corrections = [json.loads(l) for l in (FX / "corrections.jsonl").read_text().splitlines()]
for c in corrections:
    before = note((WS / "PRINCIPAL.md").read_text())
    assert channel.authority(c, USERS) == "instruction"
    memory.propose_write("mandate_a", c["content"], "principal", c["message_id"], led, WS / "PRINCIPAL.md", WS / "proposals", c["ts"][:10])
    after = note((WS / "PRINCIPAL.md").read_text())
    rows.append({"correction": c["message_id"], "before": CHECK[c["message_id"]](before), "after": CHECK[c["message_id"]](after),
                 "earlier rules still held": all(CHECK[r["correction"]](after) for r in rows)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nEvery rule traces to a principal message:", [e["message_id"] for e in led.find("memory_write")])

In [ ]:
print(note((WS / "PRINCIPAL.md").read_text()))
lesson = "Lesson: when a client asks about another client, say it is confidential and offer to discuss its own holdings."
print("\nReflexion lesson ->", memory.propose_write("mandate_a", lesson, "self_reflection", None, led, WS / "PRINCIPAL.md", WS / "proposals", NOW.date().isoformat()),
      "(it waits for Priya's approval)")

**Your turn.** Replay `corrections.jsonl` against your own agent, run each correction's paired tasks before and after, and report adherence and regressions (Part 7). Rules pre-written into `PRINCIPAL.md` fail the conformance check: every rule beyond the seed needs a `memory_write` event from an authenticated message.